## Model 2: generating the DBQA data for Er$^{3+}$:LiYF$_4$

**Question:** how does the multiscale structure of the free-ion and crystal-field Hamiltonian affect DBQA convergence, and how costly is this dense Hamiltonian to compile compared with the spin chains studied so far?

This notebook runs the DBQA iterations and saves the data; `Example_6_DBQA_Er_LiYF4_results.ipynb` loads and analyses them. The full run takes about 1.5 hours on a laptop.

We compare the same rotations as for Model 1, with the step chosen at every iteration by a grid search of the off-diagonal norm (`spectral_grid_search_step`, the same minimisation as `grid_search_step` but cheaper at 512 dimensions):
- `canonical` bracket,
- `Pauli-Z`: single commutator with $D_k$ the best single-qubit $Z_i$ (both signs), i.e. the magnetic-field generators of DB-DOI; all 511 $Z$ products are too many at this size,
- `group commutator` with $D_k=\Delta(\hat H_k)$.

At every iteration we record the spectrum report for the Kramers doublets and for the manifolds defined in `Example_4`.

In [1]:
import os
import csv
import logging
import pickle

import numpy as np
import matplotlib.pyplot as plt

from qibo import set_backend
from qibo.config import log

from pycf import cfl
from pycf.import_sljm import ImportSLJM

from qibo_dbqa.utils_crystal_field import *

# PyCF source tree with the bundled matrix-element bases (examples/*/matel)
os.environ.setdefault("PYCF_ROOT", os.path.abspath("../../../pycf"))

set_backend("numpy")
log.setLevel(logging.ERROR)
os.makedirs("pdfs_figures", exist_ok=True)
os.makedirs("simulation_results", exist_ok=True)

import time
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp

from qibo_dbqa.utils import generate_pauli_operator_dict

[Qibo 0.2.14|INFO|2026-10-16 05:58:30]: Using numpy backend on /CPU:0


In [2]:
f11cf = ImportSLJM(pycf_matel_path("f11cf", "eryso"))
coeff = er_liyf4_pycf_coeff(ER_LIYF4_FREE_ION_REFIT)
tensors = [getattr(f11cf, name) for name in coeff]

energy_unit = 20000.0  # cm^-1, so that the norm of the embedded Hamiltonian is O(1)
h, n_physical = crystal_field_hamiltonian(tensors, coeff, energy_unit=energy_unit)
h_matrix = h.backend.to_numpy(h.matrix)
energies = np.linalg.eigvalsh(h_matrix)

# levels: Kramers doublets, grouped with 1e-3 cm^-1
tol_doublets = 1e-3 / energy_unit
doublets = group_levels(energies, tol_doublets)
# manifolds: clusters separated by more than 300 cm^-1, larger than every gap inside the ground multiplet
tol_manifolds = 300.0 / energy_unit
manifolds = group_levels(energies, tol_manifolds)
# clusters: levels closer than 1 cm^-1 merged, for the close pairs (e.g. the two lowest levels, 0.36 cm^-1 apart)
tol_clusters = 1.0 / energy_unit
clusters = group_levels(energies, tol_clusters)
inside, between = check_grouping_tolerance(energies, tol_doublets)
assert inside < tol_doublets < between
print(f"{len(doublets)} level groups, {len(clusters)} clusters, {len(manifolds)} manifolds (including the padding)")

183 level groups, 181 clusters, 36 manifolds (including the padding)


#### Stage 1: DBQA on the full Hamiltonian

The three methods run in parallel processes, each limited to a share of the BLAS threads.

In [3]:
NSTEPS = 150
space = np.logspace(-4, 1, 100)
d_list = [np.real(op) for op in generate_pauli_operator_dict(h.nqubits, parameterization_order=1).values()]
methods = ["canonical", "Pauli-Z", "group commutator"]
spaces = {"canonical": space, "Pauli-Z": np.logspace(-4, 1, 30), "group commutator": space}
checkpoints = tuple(range(0, NSTEPS + 1, 10))

os.environ["OPENBLAS_NUM_THREADS"] = str(max(1, os.cpu_count() // len(methods)))
start = time.time()
with ProcessPoolExecutor(max_workers=len(methods), mp_context=mp.get_context("spawn")) as pool:
    futures = {
        method: pool.submit(
            run_dbqa_history, h_matrix, method, dict(doublets=doublets, clusters=clusters, manifolds=manifolds), NSTEPS, spaces[method],
            d_list if method == "Pauli-Z" else None, checkpoints,
        )
        for method in methods
    }
    runs = {method: future.result() for method, future in futures.items()}
print(f"stage 1 finished in {(time.time() - start) / 60:.0f} min")

stage 1 finished in 35 min


#### Stage 2: DBQA inside the ground multiplet

Stage 1 isolates the free-ion manifolds long before it resolves the crystal-field levels inside them. A multiscale strategy therefore restricts the second stage to the isolated ground manifold: once the 16 basis states with the lowest diagonal entries of $\hat H_k$ span the ground multiplet $^4I_{15/2}$ to within $1-F<10^{-6}$, the $16\times16$ block of $\hat H_k$ on these states is a 4-qubit Hamiltonian whose spectrum is the crystal-field splitting of $^4I_{15/2}$. We run the Model 1 strategies on this block, in units of 100 cm$^{-1}$ so that its norm is comparable to that of Model 1 (the multiplet spans 339 cm$^{-1}$) and the same step grids apply. Errors are measured against the eigenvalues of the block; their distance to the exact levels of the full Hamiltonian is reported separately.

In [4]:
from qibo_dbqa.utils import generate_Z_operators

def ground_manifold_block(matrix, size=16):
    """Block of the iterated Hamiltonian on the `size` basis states with the lowest diagonal entries."""
    index = np.sort(np.argsort(np.real(np.diag(matrix)))[:size])
    return matrix[np.ix_(index, index)]

NSTEPS_BLOCK = 100
# the ground multiplet spans 339 cm^-1: in units of 100 cm^-1 the block has a norm comparable to Model 1
block_unit = 100.0
ground_levels_exact = energies[np.concatenate(doublets[:8])]
z_ops_block = [np.real(op) for op in generate_Z_operators(4).values()]
stage2 = {}
for method, run in runs.items():
    infidelities = [1 - r["manifolds"]["level_fidelity"][0] for r in run["history"]]
    candidates = [k for k in sorted(run["checkpoints"]) if infidelities[k] < 1e-6]
    if not candidates:
        stage2[method] = None
        print(f"{method}: ground manifold not isolated to 1-F < 1e-6 within {NSTEPS} iterations")
        continue
    k_switch = candidates[0]
    block = ground_manifold_block(run["checkpoints"][k_switch])
    block_cm = energy_unit * (block - np.trace(block) / 16 * np.eye(16)) / block_unit
    block_levels = np.linalg.eigvalsh(block_cm)
    leakage = np.max(np.abs(energy_unit * (np.linalg.eigvalsh(block) - ground_levels_exact)))
    groups_block = group_levels(block_levels, 1e-3 / block_unit)
    clusters_block = group_levels(block_levels, 1.0 / block_unit)
    stage2[method] = dict(k_switch=k_switch, leakage_error=leakage, block_levels=block_levels, runs={})
    for block_method in methods:
        stage2[method]["runs"][block_method] = run_dbqa_history(
            block_cm, block_method, dict(doublets=groups_block, clusters=clusters_block), NSTEPS_BLOCK,
            np.linspace(1e-4, 0.3, 300) if block_method != "group commutator" else np.linspace(1e-3, 1.0, 300),
            z_ops_block if block_method == "Pauli-Z" else None,
        )
    print(f"{method}: switch after {k_switch} iterations, block eigenvalues differ from the exact ground multiplet by at most {leakage:.2e} cm^-1")

canonical: ground manifold not isolated to 1-F < 1e-6 within 150 iterations
Pauli-Z: ground manifold not isolated to 1-F < 1e-6 within 150 iterations


group commutator: switch after 30 iterations, block eigenvalues differ from the exact ground multiplet by at most 1.76e-02 cm^-1


#### Saving the data

In [5]:
with open("simulation_results/Er_LiYF4_DBQA_L_9.pickle", "wb") as file:
    pickle.dump(
        dict(
            energy_unit=energy_unit,
            block_unit=block_unit,
            energies=energies,
            doublets=doublets,
            clusters=clusters,
            manifolds=manifolds,
            methods=methods,
            nsteps=NSTEPS,
            stage1={method: dict(history=run["history"], steps=run["steps"], generators=run["generators"]) for method, run in runs.items()},
            stage2=stage2,
        ),
        file,
    )
# the iterated matrices are large (4 MB each) and are kept out of version control
np.savez_compressed(
    "simulation_results/Er_LiYF4_DBQA_checkpoints.npz",
    **{f"{method}_{k}": matrix for method, run in runs.items() for k, matrix in run["checkpoints"].items()},
)